# Lab 2 — Learning Loop, Preprocessing, and Geometry
**Coverage:** Chapters 4–5

This notebook is one of the ten course labs. Complete the core activities in order; transfer activities are optional extensions inside the same lab and do not create additional lab numbers.


## Part A — End-to-end insurance learning loop
**Core activity.**


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
train_path = PROJECT_ROOT / "all_datasets" / "medical_cost_personal_dataset" / "insurance.csv"
df = pd.read_csv(train_path)

In [ ]:
X = df.drop(columns="charges")
y = df["charges"]

num_cols = ["age", "bmi", "children"]
cat_cols = ["sex", "smoker", "region"]

In [ ]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

In [ ]:
model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", LinearRegression()),
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)
model.fit(X_train, y_train)
pred = model.predict(X_valid)

In [ ]:
print("Validation MAE:", round(mean_absolute_error(y_valid, pred), 2))
print("Validation RMSE:", round(np.sqrt(mean_squared_error(y_valid, pred)), 2))

## Part B — Breast-cancer geometry and scaling
**Core activity.**


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
train_path = PROJECT_ROOT / "all_datasets" / "breast_cancer_dataset" / "data.csv"
df = pd.read_csv(train_path)

In [ ]:
feature_cols = [c for c in df.columns if c not in {"id", "diagnosis", "Unnamed: 32"}]
X = df[feature_cols]

In [ ]:
# Compare raw Euclidean distance with distance after standardization.
raw = pairwise_distances(X.iloc[:5], metric="euclidean")
X_scaled = StandardScaler().fit_transform(X)
scaled = pairwise_distances(X_scaled[:5], metric="euclidean")

In [ ]:
np.set_printoptions(precision=2, suppress=True)
print("Raw distances among first five rows:\n", raw)
print("\nScaled distances among first five rows:\n", scaled)

In [ ]:
# Cosine similarity answers a different question: direction rather than magnitude.
cosine_distance = pairwise_distances(X_scaled[:5], metric="cosine")
print("\nCosine similarities:\n", 1 - cosine_distance)